<a href="https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 Hypothesis (staleness_days): Content that has not been refreshed in a long time loses visibility and traffic over time as search engines favor fresher alternatives.

Signal 2 Hypothesis (ctr_vs_expected): Pages where CTR is significantly lower than expected for their average position are losing clicks due to weak titles or irrelevant snippets.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# 1. Self-Healing Data Loader

candidate_files = (
    glob.glob('../**/*.parquet', recursive=True) +
    glob.glob('./**/*.parquet', recursive=True) +
    glob.glob('../**/*.csv', recursive=True) +
    glob.glob('./**/*.csv', recursive=True)
)

# Exclude output files or git-ignored files
data_files = [f for f in candidate_files if 'outputs' not in f and '.git' not in f]

if not data_files:
    raise FileNotFoundError("No .parquet or .csv data files found in the repository directory!")

data_path = data_files[0]
print(f"Successfully located dataset at: {data_path}")

# Load file based on extension
if data_path.endswith('.parquet'):
    df = pd.read_parquet(data_path)
else:
    df = pd.read_csv(data_path)

print(f"Loaded dataset with shape: {df.shape}")


# 2. Dynamic Column Mapping

def get_col(candidates, fallback):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback

col_url = get_col(['url', 'page', 'uri', 'id'], 'url')
col_stale = get_col(['staleness_days', 'staleness', 'days_since_refresh', 'age_days'], 'staleness_days')
col_ctr_gap = get_col(['ctr_vs_expected', 'ctr_gap', 'ctr_delta', 'ctr_diff'], 'ctr_vs_expected')
col_target = get_col(['target_traffic_delta', 'traffic_delta', 'target', 'traffic_growth', 'clicks_delta'], 'target_traffic_delta')

print(f"Mapped columns -> URL: '{col_url}', Staleness: '{col_stale}', CTR Gap: '{col_ctr_gap}', Target: '{col_target}'")

# Standardize dataset columns internally
df = df.rename(columns={
    col_url: 'url',
    col_stale: 'staleness_days',
    col_ctr_gap: 'ctr_vs_expected',
    col_target: 'target_traffic_delta'
})

# Safe fallback for missing target metric during initial exploration
if 'target_traffic_delta' not in df.columns:
    df['target_traffic_delta'] = 0.0

# 3. Signal 1 Check: Staleness Days (FlyRank Flag Signal)

df['staleness_bucket'] = pd.qcut(df['staleness_days'], q=5, duplicates='drop')

s1_table = df.groupby('staleness_bucket', observed=False).agg(
    n=('url', 'count'),
    mean_target=('target_traffic_delta', 'mean'),
    median_target=('target_traffic_delta', 'median')
).reset_index()

print("\n=========================================================")
print(" SIGNAL 1 CHECK: Staleness Days (Refresh Logic)")
print("=========================================================")
print(s1_table.to_string(index=False))

s1_corr = df['staleness_days'].corr(df['target_traffic_delta'])
print(f"\nCorrelation with target: {s1_corr:.4f}")

if s1_corr < -0.05:
    s1_verdict = "CONFIRMED"
    s1_reason = "Traffic performance drops consistently as content staleness increases."
elif s1_corr > 0.05:
    s1_verdict = "OPPOSITE"
    s1_reason = "Older content shows higher performance, contradicting refresh hypothesis."
elif abs(s1_corr) <= 0.05:
    s1_verdict = "MIXED"
    s1_reason = "No clear monotonic trend across staleness buckets."
else:
    s1_verdict = "FALSE"
    s1_reason = "Signal shows zero predictive value."

print(f"\n>>> VERDICT 1: {s1_verdict} — {s1_reason}\n")


# 4. Signal 2 Check: CTR vs Expected (Position-Adjusted CTR)

df['ctr_gap_bucket'] = pd.qcut(df['ctr_vs_expected'], q=5, duplicates='drop')

s2_table = df.groupby('ctr_gap_bucket', observed=False).agg(
    n=('url', 'count'),
    mean_target=('target_traffic_delta', 'mean'),
    median_target=('target_traffic_delta', 'median')
).reset_index()

print("=========================================================")
print(" SIGNAL 2 CHECK: CTR vs Expected (CTR-Fix Logic)")
print("=========================================================")
print(s2_table.to_string(index=False))

s2_corr = df['ctr_vs_expected'].corr(df['target_traffic_delta'])
print(f"\nCorrelation with target: {s2_corr:.4f}")

if s2_corr > 0.05:
    s2_verdict = "CONFIRMED"
    s2_reason = "Pages underperforming on CTR suffer higher traffic loss."
elif s2_corr < -0.05:
    s2_verdict = "OPPOSITE"
    s2_reason = "Negative CTR gap correlates with higher target traffic."
elif abs(s2_corr) <= 0.05:
    s2_verdict = "MIXED"
    s2_reason = "CTR gap performance varies non-linearly across position tiers."
else:
    s2_verdict = "FALSE"
    s2_reason = "Signal has no predictive relationship with future traffic."

print(f"\n>>> VERDICT 2: {s2_verdict} — {s2_reason}")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import pandas as pd
import numpy as np

def apply_baseline_rule(df_in):

    df_out = df_in.copy()

    # Calculate components
    staleness_factor = (df_out['staleness_days'] / 30.0).clip(lower=0)
    ctr_deficit = (-df_out['ctr_vs_expected']).clip(lower=0)

    # Vectorized condition masks
    cond_stale = df_out['staleness_days'] > 180
    cond_low_ctr = df_out['ctr_vs_expected'] < -0.02

    # Assign Reason Codes and Action Labels
    conditions = [
        cond_stale & cond_low_ctr,
        cond_stale & ~cond_low_ctr,
        ~cond_stale & cond_low_ctr
    ]

    reasons = [
        'STALE_AND_LOW_CTR',
        'STALE_CONTENT_ONLY',
        'LOW_CTR_ONLY'
    ]

    actions = [
        'REFRESH_AND_OPTIMIZE_TITLE',
        'REFRESH_CONTENT',
        'OPTIMIZE_TITLE_SNIPPET'
    ]

    df_out['reason_code'] = np.select(conditions, reasons, default='NO_ACTION_REQUIRED')
    df_out['action_label'] = np.select(conditions, actions, default='MAINTAIN')

    # Compute continuous priority score
    scores = np.select(
        conditions,
        [
            staleness_factor * (1.0 + (ctr_deficit * 100.0)),
            staleness_factor * 0.5,
            ctr_deficit * 50.0
        ],
        default=0.0
    )

    df_out['action_score'] = scores
    return df_out

# Run rule scoring & generate ranked queue
df_scored = apply_baseline_rule(df)
df_ranked = df_scored.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# Write output CSV (Git-ignored work output)
os.makedirs('../outputs', exist_ok=True)
output_cols = ['url', 'action_score', 'reason_code', 'action_label']
output_path = '../outputs/baseline_action_score.csv'

df_ranked[output_cols].to_csv(output_path, index=False)

print(f"Ranked queue written to {output_path} ({len(df_ranked)} rows processed).")
print(f"Top action counts:\n{df_ranked['action_label'].value_counts()}")

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Inspect Top 20 Recommendations
top_20 = df_ranked.head(20)

print("=== TOP 20 QUEUE REVIEW ===")
for idx, row in top_20.iterrows():
    # Build inline skeptic justification based on signal flags
    if row['reason_code'] == 'STALE_AND_LOW_CTR':
        confidence = "HIGH"
        wrong_if = "Page targets an intentional zero-click query, or SERP features (featured snippets/PPA) absorb clicks."
    elif row['reason_code'] == 'STALE_CONTENT_ONLY':
        confidence = "MEDIUM"
        wrong_if = "Content covers evergreen topics (e.g., historical definitions) where information remains 100% accurate."
    else:
        confidence = "LOW"
        wrong_if = "Recent title change hasn't had sufficient SERP impression volume to establish stable CTR."

    print(f"Rank {idx+1:02d} | URL: {row['url']}")
    print(f"  Action: {row['action_label']} | Reason: {row['reason_code']} | Score: {row['action_score']:.2f}")
    print(f"  Confidence: {confidence} | What would make it wrong: {wrong_if}\n")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Part 1: Identify Weak Picks in Top Priority Queue
weak_picks = df_ranked[
    (df_ranked['action_score'] > 0) &
    (df_ranked['impressions'] < 100) # Low volume noise
].head(3)

print("=== WEAK PICKS AUDIT ===")
if len(weak_picks) > 0:
    for idx, row in weak_picks.iterrows():
        print(f"Weak Pick URL: {row['url']}")
        print(f"  Issue: High score ({row['action_score']:.2f}) driven by high variance CTR on low impressions ({row['impressions']}).")
else:
    print("No weak low-volume picks detected in top bucket.")

print("\n=== LEAKAGE VERIFICATION ===")


forbidden_patterns = ['flag_', 'future_', 'target_', 'label_', 'next_period', 'post_']

df_cols = list(df.columns)
input_cols_used = ['staleness_days', 'ctr_vs_expected', 'impressions']

leaked_cols = [
    col for col in input_cols_used
    if any(pattern in col.lower() for pattern in forbidden_patterns)
]

# Assert no future window or label leakage
assert len(leaked_cols) == 0, f"LEAKAGE DETECTED: {leaked_cols}"

print(f"Columns evaluated for rule: {input_cols_used}")
print("LEAKAGE CHECK PASSED: Zero product flags, target labels, or future-window signals used.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.